# NSE Demerger PDF Downloader (Colab)
Downloads **~4,648** demerger documents (PDFs + ZIPs) from NSE → Google Drive.

- ZIPs are auto-extracted; only PDFs are kept.
- Full resume support — re-run anytime, nothing re-downloaded.

**Run Cell 0 → 1 → 2 → 3 → 4**

In [ ]:
# CELL 0: Keepalive — prevents Colab from disconnecting
import threading, time
def _ka():
    while True: time.sleep(55); _ = 1+1
if not any(t.name=='keepalive' for t in threading.enumerate()):
    threading.Thread(target=_ka, name='keepalive', daemon=True).start()
    print('✅ Keepalive started')
else:
    print('✅ Keepalive already running')

In [ ]:
# CELL 1: Mount Drive (skips if already mounted)
import os, shutil, time
from google.colab import drive
MOUNT = '/content/drive'
already_ok = False
try:
    tp = os.path.join(MOUNT, 'MyDrive')
    if os.path.isdir(tp) and os.listdir(tp): already_ok = True
except: pass
if already_ok:
    print('✅ Drive already mounted — skipping.')
else:
    if os.path.isdir(MOUNT):
        try: shutil.rmtree(MOUNT, ignore_errors=True)
        except: pass
    for attempt in range(1, 6):
        try:
            drive.mount(MOUNT)
            print('✅ Drive mounted!')
            break
        except Exception as e:
            if 'already contain' in str(e): shutil.rmtree(MOUNT, ignore_errors=True)
            if attempt < 5: print(f'Mount {attempt}/5 failed: {e}, retry in {10*attempt}s'); time.sleep(10*attempt)
            else: raise

In [ ]:
# CELL 2: Locate nse_demerger.csv — hardcoded path for speed
import os

# ✅ Direct path — no slow os.walk needed
MANIFEST_CSV = '/content/drive/MyDrive/ArthLLM-2B/demerger/nse_demerger.csv'

if not os.path.exists(MANIFEST_CSV):
    raise FileNotFoundError(f'CSV not found: {MANIFEST_CSV}\nCheck the path in My Drive.')

BASE    = os.path.dirname(MANIFEST_CSV)
OUT_DIR = os.path.join(BASE, 'NSE_Demerger', 'raw_pdfs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'✅ Manifest : {MANIFEST_CSV}')
print(f'📥 Output   : {OUT_DIR}')

In [ ]:
# CELL 3: Fast Resume — sync manifest against already-downloaded files
import os, re
import pandas as pd
from tqdm.notebook import tqdm

# ── Load CSV ──────────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype={'seq_id': str})
except Exception as e:
    raise RuntimeError(f'CORRUPT CSV: {e}')
if len(df) == 0: raise RuntimeError('CSV has 0 rows')
print(f'Loaded {len(df)} rows')

# ── Normalise status column ───────────────────────────────────────────────────
DONE_VALS = ('done', 'skip', 'true')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
                  else (str(v).strip().lower() if pd.notna(v) and str(v).strip() != '' else 'pending')
    )

# ── Rows with no URL → mark skip so they're never attempted ───────────────────
no_url_mask = df['pdf_url'].isna() | (df['pdf_url'].astype(str).str.strip() == '-')
df.loc[no_url_mask, 'dl_status'] = 'no_url'
print(f'  Rows with no URL : {no_url_mask.sum()}')

# ── Scan existing PDFs ────────────────────────────────────────────────────────
print('Scanning for existing PDFs...')
existing = set()
if os.path.exists(OUT_DIR):
    for r, d, files in os.walk(OUT_DIR):
        for x in files:
            if x.lower().endswith('.pdf'): existing.add(x)
print(f'Found {len(existing)} PDFs on disk')

# ── Helper: derive expected filename for a row ────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def expected_fname(row):
    sym  = safe(row.get('symbol', 'UNKNOWN'))
    seq  = safe(str(row.get('seq_id', '')).split('.')[0])
    url  = str(row.get('pdf_url', ''))
    base = os.path.splitext(os.path.basename(url))[0][:40]
    return f'{sym}_{seq}_{base}.pdf'

# ── Two-way sync ──────────────────────────────────────────────────────────────
updated = 0
for i in tqdm(df.index, desc='Syncing manifest'):
    if df.at[i, 'dl_status'] == 'no_url': continue
    cur = df.at[i, 'dl_status']
    fn  = expected_fname(df.loc[i])
    fe  = fn in existing
    if cur == 'done' and not fe: df.at[i, 'dl_status'] = 'pending'; updated += 1
    elif cur != 'done' and fe:   df.at[i, 'dl_status'] = 'done';    updated += 1

if updated > 0:
    df.to_csv(MANIFEST_CSV, index=False, quoting=1)
    print(f'Updated {updated} rows in manifest')
else:
    print('Manifest already in sync')

counts = df['dl_status'].value_counts()
print('\nStatus summary:')
for s, c in counts.items(): print(f'  {s:12s}: {c}')

In [ ]:
# CELL 4: Download NSE Demerger Docs (PDFs + ZIPs auto-extracted)
import pandas as pd, requests, os, re, time, threading, zipfile, io
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
WORKERS    = 12    # concurrent threads (NSE rate-limits; 12 is safe)
SAVE_EVERY = 100   # save manifest every N completions
TIMER_SAVE = 240   # also save every 4 minutes
TIMEOUT    = 45    # per-request timeout (seconds)
MAX_RETRIES = 4

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'application/pdf,application/zip,*/*',
    'Referer': 'https://www.nseindia.com/'
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def is_valid_pdf(data):
    if not data or len(data) < 64: return False
    if data[:4] == b'%PDF': return True
    if b'<html' in data[:512].lower() or b'<!doctype' in data[:512].lower(): return False
    return len(data) > 2048

def extract_pdfs_from_zip(zip_bytes, dest_dir, base_name):
    """Extract all PDFs from a zip. Returns list of saved filenames."""
    saved = []
    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            pdf_members = [m for m in zf.namelist() if m.lower().endswith('.pdf')]
            if not pdf_members:
                # Sometimes zips contain more zips — go one level deep
                for m in zf.namelist():
                    if m.lower().endswith('.zip'):
                        inner = zf.read(m)
                        saved += extract_pdfs_from_zip(inner, dest_dir, base_name + '_inner')
                return saved
            for idx, member in enumerate(pdf_members):
                pdf_data = zf.read(member)
                suffix = f'_{idx+1}' if len(pdf_members) > 1 else ''
                fname = f'{base_name}{suffix}.pdf'
                out_path = os.path.join(dest_dir, fname)
                os.makedirs(dest_dir, exist_ok=True)
                with open(out_path, 'wb') as f: f.write(pdf_data)
                saved.append(fname)
    except Exception as e:
        pass  # corrupt zip → treated as fail
    return saved

def download_row(url, dest_dir, base_name):
    """
    Returns: 'skip' | 'ok' | 'fail' | 'http_NNN'
    For ZIPs: extracts PDFs; returns 'ok' if at least one PDF saved.
    """
    is_zip = url.lower().endswith('.zip')
    dest_pdf = os.path.join(dest_dir, f'{base_name}.pdf')

    # Skip if already present (PDF direct)
    if not is_zip and os.path.exists(dest_pdf) and os.path.getsize(dest_pdf) > 1024:
        return 'skip'
    # Skip if already present (from zip — check _1.pdf)
    if is_zip and os.path.exists(os.path.join(dest_dir, f'{base_name}_1.pdf')):
        return 'skip'
    if is_zip and os.path.exists(dest_pdf):  # single-pdf zip already done
        return 'skip'

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if r.status_code == 200:
                data = r.content
                if is_zip:
                    os.makedirs(dest_dir, exist_ok=True)
                    saved = extract_pdfs_from_zip(data, dest_dir, base_name)
                    return 'ok' if saved else 'fail'
                else:
                    if not is_valid_pdf(data): return 'bad_content'
                    os.makedirs(dest_dir, exist_ok=True)
                    with open(dest_pdf, 'wb') as f: f.write(data)
                    return 'ok'
            if r.status_code in (429, 503):
                time.sleep(8 * (2 ** attempt)); continue
            return f'http_{r.status_code}'
        except requests.exceptions.Timeout:
            time.sleep(3 * (attempt + 1))
        except Exception:
            time.sleep(2 * (attempt + 1))
    return 'fail'

# ── Load manifest ─────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype={'seq_id': str})
except Exception as e:
    raise RuntimeError(f'CORRUPT CSV: {e}')

DONE_VALS = ('done', 'skip', 'true')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
                  else (str(v).strip().lower() if pd.notna(v) and str(v).strip() != '' else 'pending')
    )

# Rows with no URL are not downloadable
no_url_mask = df['pdf_url'].isna() | (df['pdf_url'].astype(str).str.strip() == '-')
df.loc[no_url_mask, 'dl_status'] = 'no_url'

# ── Build task list ───────────────────────────────────────────────────────────
pending = df[~df['dl_status'].isin(['done', 'no_url'])]
print(f'Pending: {len(pending)} / {len(df)}  (no_url: {no_url_mask.sum()})')

if len(pending) == 0:
    print('🎉 All done! Nothing to download.')
else:
    tasks = []
    for idx, row in pending.iterrows():
        url     = str(row['pdf_url']).strip()
        sym     = safe(row.get('symbol', 'UNKNOWN'))
        seq     = safe(str(row.get('seq_id', '')).split('.')[0])
        url_base = os.path.splitext(os.path.basename(url))[0][:40]
        base_name = f'{sym}_{seq}_{url_base}'
        dest_dir  = os.path.join(OUT_DIR, sym)  # one folder per symbol
        tasks.append((idx, url, dest_dir, base_name))

    ok = skip = fail = counter = 0
    _stop = threading.Event()
    _lock = threading.Lock()

    def _autosave():
        while not _stop.wait(TIMER_SAVE):
            with _lock:
                try: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
                except: pass
    threading.Thread(target=_autosave, daemon=True, name='autosave').start()

    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            futs = {pool.submit(download_row, u, d, b): (i, u) for i, u, d, b in tasks}
            pbar = tqdm(total=len(futs), desc='NSE Demerger Docs', unit='file')
            for fut in as_completed(futs):
                i, u = futs[fut]
                try:    res = fut.result()
                except: res = 'fail'
                if res == 'ok':   ok   += 1; df.at[i, 'dl_status'] = 'done'
                elif res == 'skip': skip += 1; df.at[i, 'dl_status'] = 'done'
                else:             fail += 1; df.at[i, 'dl_status'] = res
                counter += 1
                if counter % SAVE_EVERY == 0:
                    with _lock: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
                pbar.set_postfix(ok=ok, skip=skip, fail=fail, refresh=False)
                pbar.update(1)
            pbar.close()
    except KeyboardInterrupt:
        print('\n⚠️  Ctrl+C — saving manifest before exit...')
    finally:
        _stop.set()
        with _lock: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
        print(f'💾 Manifest saved: {MANIFEST_CSV}')

    total  = len(df)
    done_n = df['dl_status'].isin(['done']).sum()
    no_url_n = (df['dl_status'] == 'no_url').sum()
    print(f'\n{'='*55}')
    print(f'  SUMMARY')
    print(f'  Total rows  : {total}')
    print(f'  Done        : {done_n}')
    print(f'  OK (new)    : {ok}')
    print(f'  Skipped     : {skip}')
    print(f'  Failed      : {fail}')
    print(f'  No URL      : {no_url_n}')
    print(f'  Still pending: {total - done_n - no_url_n}')
    print(f'{'='*55}')